# 01.4 Compiled vs Interpreted Languages

Python is usually called "interpreted", but it is really a hybrid:
it COMPILES your source to bytecode, then INTERPRETS that bytecode.

This file proves both halves of that sentence.


## PART 1: The compile step really happens


In [ ]:
# The compile() built-in does exactly what Python does to your file
# automatically. We can call it by hand to watch the step happen.

# A tiny piece of Python source, written as text.
source_text = "total = 10 + 20"

# compile() turns that text into a code object full of bytecode.
# The arguments are: the source, a filename label, and the mode.
compiled_code = compile(source_text, "<example>", "exec")

# The result is a real object, not text any more.
print("Source text was:", repr(source_text))
print("After compiling, we have a:", type(compiled_code).__name__)
print("Its bytecode is raw bytes:", compiled_code.co_code)

# Those bytes are instructions for the Python Virtual Machine - not for
# your CPU. That is the crucial difference from a compiled language,
# where the output would be native machine code.

## PART 2: Reading the bytecode


In [ ]:
# The dis module translates those raw bytes into readable instructions.
import dis

print("Bytecode for:", repr(source_text))

# dis.dis() prints one line per instruction, in execution order.
dis.dis(compiled_code)

print("Notice Python already did the addition at compile time - it saw")
print("10 + 20 were both constants and stored 30 directly.")
print("That optimisation is called constant folding.")

## PART 3: Compile-time errors vs runtime errors


In [ ]:
# Because compilation happens BEFORE execution, syntax errors are found
# immediately - the whole file fails to compile and nothing runs.

# This source has a syntax error: a missing closing bracket.
broken_syntax = "result = (10 + 20"

# We use try/except (Chapter 23) to catch the failure instead of crashing.
try:
    # compile() fails here, before a single instruction is executed.
    compile(broken_syntax, "<example>", "exec")
except SyntaxError as error:
    print("SYNTAX ERROR - caught at COMPILE time, before anything ran:")
    print("   ", error.msg)

# Other errors survive compilation and only appear when that line runs.
# This source is perfectly valid Python - it just will not work.
valid_but_wrong = "result = undefined_name + 1"

# Compiling succeeds, because the syntax is fine.
compiled_wrong = compile(valid_but_wrong, "<example>", "exec")
print("This source compiled fine:", repr(valid_but_wrong))

# The problem only surfaces when we actually execute it.
try:
    # exec() runs a compiled code object.
    exec(compiled_wrong)
except NameError as error:
    print("NAME ERROR - caught at RUNTIME, only when the line executed:")
    print("   ", error)

print("This is the interpreted-language trade-off: Python will happily")
print("run 399 correct lines before discovering the mistake on line 400.")
print("Tests (Chapter 36) and type hints (Chapter 32) close that gap.")

## PART 4: The __pycache__ folder


In [ ]:
# When Python imports a MODULE, it saves the compiled bytecode to disk
# so it does not have to recompile next time. That cache is __pycache__.

# We import sys to inspect how Python is configured.
import sys

# This flag tells us whether Python is allowed to write .pyc cache files.
print("Is bytecode caching enabled?", not sys.dont_write_bytecode)

# The cache tag identifies which interpreter version wrote the files,
# so different Python versions never read each other's bytecode.
print("Cache tag for this interpreter:", sys.implementation.cache_tag)

print("Facts worth knowing about __pycache__:")
print("  - It appears next to modules you IMPORT, not scripts you RUN.")
print("  - It holds .pyc files: the compiled bytecode, cached.")
print("  - It is safe to delete - Python regenerates it automatically.")
print("  - It belongs in .gitignore, never in version control.")

## PART 5: Why interpreted is slower, measured


In [ ]:
import time

# In a compiled language, `a + b` becomes one CPU instruction.
# In Python, every operation goes through the interpreter, which must
# check types at runtime before it knows what "+" even means here.

# Time a simple loop doing one million additions.
start = time.perf_counter()

total = 0
for number in range(1000000):
    # Each pass: fetch values, check their types, dispatch to the right
    # addition, allocate a result object, store it. Not one instruction.
    total = total + number

interpreted_duration = time.perf_counter() - start

print("One million additions in a Python loop:")
print("   ", round(interpreted_duration * 1000, 2), "milliseconds")

# The same work handed to sum(), which runs its loop in C rather than
# in interpreted Python bytecode.
start = time.perf_counter()
total_via_builtin = sum(range(1000000))
builtin_duration = time.perf_counter() - start

print("The same work via the built-in sum() (a C loop):")
print("   ", round(builtin_duration * 1000, 2), "milliseconds")

# Guard against a division by zero on very fast machines.
if builtin_duration > 0:
    speedup = interpreted_duration / builtin_duration
    print("   the C loop was about", round(speedup, 1), "times faster")

print("Lesson: push hot loops into C-backed built-ins and libraries.")
print("This is exactly how NumPy makes Python fast for numeric work.")

## PART 6: Comparing the two models


In [ ]:
# Printed as plain lines so the comparison is easy to read in a terminal.
print("COMPILED (C, C++, Rust, Go):")
print("   translate everything up front, then run the machine code")
print("   + fast at runtime")
print("   + errors caught before shipping")
print("   - must rebuild after every change")
print("   - the built file only runs on its target platform")
print("INTERPRETED (Python, JavaScript, Ruby):")
print("   translate and execute as it goes")
print("   + change a line and run it instantly")
print("   + the same file runs anywhere the interpreter exists")
print("   - slower at runtime")
print("   - some errors only appear when that line finally runs")
print("PYTHON is both: source -> bytecode (compiled),")
print("                bytecode -> executed by the PVM (interpreted).")

## TAKEAWAYS


In [ ]:
print("TAKEAWAYS")
print("1. Python compiles source to bytecode, then interprets the bytecode.")
print("2. Bytecode targets the Python VM, not your CPU.")
print("3. Syntax errors are caught at compile time, before anything runs.")
print("4. Name and type errors wait until that line actually executes.")
print("5. __pycache__ caches bytecode; it is safe to delete and to ignore.")
print("6. Interpreting costs speed - push hot loops into C-backed code.")

## TRY IT YOURSELF

1. Change source_text to "total = 10 * 20 + 5" and rerun. Did Python
   fold all of it into one constant?

2. Change it to "total = x + 20". Now the value is unknown at compile
   time - how does the bytecode differ?

3. Run `python -X importtime -c "import json"` in your terminal.
   Run it twice. Why is the second run faster?

4. Deliberately put a syntax error at the very BOTTOM of this file and
   rerun. Does any of the output print? Explain why not.
